In [5]:
pip install requests pymysql python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\yoona\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [1]:
MYSQL_HOST=localhost
MYSQL_PORT=3306
MYSQL_USER=root
MYSQL_PASSWORD=mysql
MYSQL_DATABASE=car_data

NameError: name 'localhost' is not defined

In [9]:
import os
import time
from datetime import datetime

import pymysql
import requests
from dotenv import load_dotenv


load_dotenv()


# =========================================================
# 기본 설정
# =========================================================

BASE_URL = "http://192.168.0.51:4000"

PUBLIC_KEY_URL = f"{BASE_URL}/api/v1/public-key"
CARS_CURSOR_URL = f"{BASE_URL}/api/v1/cars/cursor"

TEST_LIMIT = 200
PAGE_LIMIT = 100

MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST"),
    "port": int(os.getenv("MYSQL_PORT", 3306)),
    "user": os.getenv("MYSQL_USER"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE"),
    "charset": "utf8mb4",
    "cursorclass": pymysql.cursors.DictCursor,
    "autocommit": False,
}


# =========================================================
# 1. API Key 조회
# =========================================================

def get_api_key():
    response = requests.get(
        PUBLIC_KEY_URL,
        timeout=10
    )

    response.raise_for_status()

    body = response.json()

    api_key = body["data"]["current"]["api_key"]

    print("[API KEY] 현재 API Key 조회 완료")

    return api_key


# =========================================================
# 2. API 호출
# =========================================================

def request_api(url, api_key):
    headers = {
        "X-API-Key": api_key
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=15
    )

    # 키가 자정에 바뀐 경우
    if response.status_code == 403:

        print("[WARN] API Key 만료 또는 변경 감지")

        new_key = get_api_key()

        headers["X-API-Key"] = new_key

        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

    response.raise_for_status()

    return response.json()


# =========================================================
# 3. 최대 200건 차량 수집
# =========================================================

def fetch_cars(limit=200):

    api_key = get_api_key()

    cars = []

    next_url = (
        f"{CARS_CURSOR_URL}"
        f"?after_id=0&limit={PAGE_LIMIT}"
    )

    while next_url and len(cars) < limit:

        print(f"[FETCH] {next_url}")

        result = request_api(
            next_url,
            api_key
        )

        data = result.get("data", [])

        if not data:
            break

        remain = limit - len(cars)

        cars.extend(data[:remain])

        print(
            f"[FETCH] 이번 조회: {len(data)}건 / "
            f"누적: {len(cars)}건"
        )

        next_url = result.get("links", {}).get("next")

        if next_url and next_url.startswith("/"):
            next_url = BASE_URL + next_url

        # 테스트 서버에 과도한 요청 방지
        time.sleep(0.3)

    return cars


# =========================================================
# 유틸
# =========================================================

def value_from(obj, *keys, default=None):
    """
    여러 후보 key 중 존재하는 첫 값을 반환.
    API 내부 object key가 정확히 확인되지 않은 부분 대응용.
    """

    if not isinstance(obj, dict):
        return default

    for key in keys:
        value = obj.get(key)

        if value is not None:
            return value

    return default


In [10]:
def normalize_car(raw):

    brand = raw.get("brand") or {}
    model = raw.get("model") or {}
    dealer = raw.get("dealer") or {}
    area = raw.get("businessArea") or {}
    location = raw.get("location") or {}

    return {
        # -----------------------
        # 핵심 식별자 4개
        # -----------------------

        "car_id": raw.get("id"),

        "listing_number": raw.get(
            "listingNumber"
        ),

        "dealer_id": value_from(
            dealer,
            "id",
            "code",
            "dealerId",
            "dealerCode"
        ),

        "business_area_code": value_from(
            area,
            "code",
            "id",
            "businessAreaCode"
        ),

        # -----------------------
        # 차량 기본 정보
        # -----------------------

        "brand": value_from(
            brand,
            "name",
            default=brand if isinstance(brand, str) else None
        ),

        "model": value_from(
            model,
            "name",
            default=model if isinstance(model, str) else None
        ),

        "trim": raw.get("trim"),

        "model_year": raw.get("modelYear"),

        "first_registration_date": (
            raw.get("firstRegistrationDate")
            or raw.get("firstRegisteredAt")
        ),

        # -----------------------
        # 제원
        # -----------------------

        "mileage_km": raw.get("mileageKm"),

        "price": raw.get("price"),

        "currency": raw.get("currency"),

        "fuel_type": (
            raw.get("fuelType")
            or raw.get("fuel")
        ),

        "transmission": raw.get("transmission"),

        "color": raw.get("color"),

        "displacement_cc": (
            raw.get("displacementCc")
            or raw.get("engineDisplacementCc")
        ),

        # -----------------------
        # 판매 상태
        # -----------------------

        "status": raw.get("status"),

        # -----------------------
        # 사고 / 소유 이력
        # -----------------------

        "accident_count": (
            raw.get("accidentCount")
            or 0
        ),

        "owner_change_count": (
            raw.get("ownerChangeCount")
            or 0
        ),

        "inspection_status": raw.get(
            "inspectionStatus"
        ),

        # -----------------------
        # 소재지
        # -----------------------

        "province": value_from(
            location,
            "province",
            "sido",
            "region"
        ),

        "city": value_from(
            location,
            "city",
            "sigungu",
            "district"
        ),

        # -----------------------
        # 매물 등록일
        # -----------------------

        "listing_date": (
            raw.get("listingDate")
            or raw.get("registeredDate")
        )
    }


# =========================================================
# 5. 테이블 생성
# =========================================================

def create_tables(conn):

    with conn.cursor() as cursor:

        # -------------------------------------------
        # 업무영역
        # -------------------------------------------

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS business_areas (

            business_area_code VARCHAR(100)
                PRIMARY KEY,

            business_area_name VARCHAR(255),

            dealer_id VARCHAR(100),

            dealer_name VARCHAR(100),

            department VARCHAR(255),

            position VARCHAR(100)

        ) ENGINE=InnoDB
        DEFAULT CHARSET=utf8mb4;
        """)

        # -------------------------------------------
        # 차량
        # -------------------------------------------

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS cars (

            car_id BIGINT PRIMARY KEY,

            listing_number VARCHAR(100)
                NOT NULL UNIQUE,

            dealer_id VARCHAR(100),

            business_area_code VARCHAR(100),

            brand VARCHAR(100),

            model VARCHAR(150),

            trim VARCHAR(150),

            model_year INT,

            first_registration_date DATE,

            mileage_km INT,

            price BIGINT,

            currency VARCHAR(20),

            fuel_type VARCHAR(50),

            transmission VARCHAR(50),

            color VARCHAR(50),

            displacement_cc INT,

            status VARCHAR(50),

            accident_count INT,

            owner_change_count INT,

            inspection_status VARCHAR(100),

            province VARCHAR(100),

            city VARCHAR(100),

            listing_date DATE,

            CONSTRAINT fk_car_business_area
                FOREIGN KEY (business_area_code)
                REFERENCES business_areas(
                    business_area_code
                )

        ) ENGINE=InnoDB
        DEFAULT CHARSET=utf8mb4;
        """)

        # -------------------------------------------
        # FAQ
        # 지금 테스트에서는 생성만 하고 적재하지 않음
        # -------------------------------------------

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS faqs (

            faq_id BIGINT AUTO_INCREMENT
                PRIMARY KEY,

            brand VARCHAR(100),

            category VARCHAR(150),

            question TEXT,

            answer TEXT,

            source_url VARCHAR(1000)

        ) ENGINE=InnoDB
        DEFAULT CHARSET=utf8mb4;
        """)

        # -------------------------------------------
        # 크롤링/수집 로그
        # -------------------------------------------

        cursor.execute("""
        CREATE TABLE IF NOT EXISTS crawl_logs (

            log_id BIGINT AUTO_INCREMENT
                PRIMARY KEY,

            source_type VARCHAR(20),

            source_name VARCHAR(255),

            started_at DATETIME,

            finished_at DATETIME,

            fetched_count INT DEFAULT 0,

            inserted_count INT DEFAULT 0,

            updated_count INT DEFAULT 0,

            failed_count INT DEFAULT 0,

            status VARCHAR(30),

            error_message TEXT

        ) ENGINE=InnoDB
        DEFAULT CHARSET=utf8mb4;
        """)

    conn.commit()


# =========================================================
# 6. business_area 저장
# =========================================================

def upsert_business_area(conn, raw):

    area = raw.get("businessArea") or {}
    dealer = raw.get("dealer") or {}

    area_code = value_from(
        area,
        "code",
        "id",
        "businessAreaCode"
    )

    if not area_code:
        return

    sql = """
    INSERT INTO business_areas (
        business_area_code,
        business_area_name,
        dealer_id,
        dealer_name,
        department,
        position
    )
    VALUES (
        %s, %s, %s, %s, %s, %s
    )
    ON DUPLICATE KEY UPDATE
        business_area_name =
            VALUES(business_area_name),
        dealer_id =
            VALUES(dealer_id),
        dealer_name =
            VALUES(dealer_name),
        department =
            VALUES(department),
        position =
            VALUES(position)
    """

    values = (
        area_code,

        value_from(
            area,
            "name",
            "businessAreaName"
        ),

        value_from(
            dealer,
            "id",
            "code",
            "dealerId",
            "dealerCode"
        ),

        value_from(
            dealer,
            "name",
            "maskedName"
        ),

        value_from(
            dealer,
            "department",
            "dept"
        ),

        value_from(
            dealer,
            "position",
            "rank"
        )
    )

    with conn.cursor() as cursor:
        cursor.execute(sql, values)


# =========================================================
# 7. 차량 UPSERT
# =========================================================

def upsert_car(conn, car):

    sql = """
    INSERT INTO cars (

        car_id,
        listing_number,
        dealer_id,
        business_area_code,

        brand,
        model,
        trim,
        model_year,
        first_registration_date,

        mileage_km,
        price,
        currency,
        fuel_type,
        transmission,
        color,
        displacement_cc,

        status,

        accident_count,
        owner_change_count,
        inspection_status,

        province,
        city,

        listing_date

    )
    VALUES (
        %s, %s, %s, %s,
        %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s, %s,
        %s,
        %s, %s, %s,
        %s, %s,
        %s
    )

    ON DUPLICATE KEY UPDATE

        dealer_id =
            VALUES(dealer_id),

        business_area_code =
            VALUES(business_area_code),

        brand =
            VALUES(brand),

        model =
            VALUES(model),

        trim =
            VALUES(trim),

        model_year =
            VALUES(model_year),

        first_registration_date =
            VALUES(first_registration_date),

        mileage_km =
            VALUES(mileage_km),

        price =
            VALUES(price),

        currency =
            VALUES(currency),

        fuel_type =
            VALUES(fuel_type),

        transmission =
            VALUES(transmission),

        color =
            VALUES(color),

        displacement_cc =
            VALUES(displacement_cc),

        status =
            VALUES(status),

        accident_count =
            VALUES(accident_count),

        owner_change_count =
            VALUES(owner_change_count),

        inspection_status =
            VALUES(inspection_status),

        province =
            VALUES(province),

        city =
            VALUES(city),

        listing_date =
            VALUES(listing_date)
    """

    values = tuple(car.values())

    with conn.cursor() as cursor:

        cursor.execute(
            "SELECT car_id FROM cars WHERE car_id = %s",
            (car["car_id"],)
        )

        exists = cursor.fetchone()

        cursor.execute(sql, values)

    return "updated" if exists else "inserted"


In [11]:
# =========================================================
# 8. 로그 저장
# =========================================================

def write_log(
    conn,
    started_at,
    finished_at,
    fetched,
    inserted,
    updated,
    failed,
    status,
    error_message=None
):

    sql = """
    INSERT INTO crawl_logs (

        source_type,
        source_name,

        started_at,
        finished_at,

        fetched_count,
        inserted_count,
        updated_count,
        failed_count,

        status,
        error_message

    )
    VALUES (
        %s, %s,
        %s, %s,
        %s, %s, %s, %s,
        %s, %s
    )
    """

    values = (
        "API",
        "AutoData Lab Cars",
        started_at,
        finished_at,
        fetched,
        inserted,
        updated,
        failed,
        status,
        error_message
    )

    with conn.cursor() as cursor:
        cursor.execute(sql, values)

    conn.commit()


# =========================================================
# 9. MAIN
# =========================================================

def main():

    started_at = datetime.now()

    fetched = 0
    inserted = 0
    updated = 0
    failed = 0

    conn = None

    try:

        conn = pymysql.connect(
            **MYSQL_CONFIG
        )

        create_tables(conn)

        print("\n==========================")
        print(" 차량 데이터 테스트 수집")
        print("==========================\n")

        raw_cars = fetch_cars(
            TEST_LIMIT
        )

        fetched = len(raw_cars)

        print(
            f"\n[INFO] 총 {fetched}건 수집 완료\n"
        )

        for index, raw in enumerate(
            raw_cars,
            start=1
        ):

            try:

                # FK 때문에 business_area 먼저 저장
                upsert_business_area(
                    conn,
                    raw
                )

                car = normalize_car(raw)

                result = upsert_car(
                    conn,
                    car
                )

                if result == "inserted":
                    inserted += 1
                else:
                    updated += 1

                conn.commit()

                print(
                    f"[{index}/{fetched}] "
                    f"{car['car_id']} "
                    f"{car['listing_number']} "
                    f"{result}"
                )

            except Exception as e:

                conn.rollback()

                failed += 1

                print(
                    f"[ERROR] "
                    f"record={index} "
                    f"{e}"
                )

        finished_at = datetime.now()

        write_log(
            conn,
            started_at,
            finished_at,
            fetched,
            inserted,
            updated,
            failed,
            "SUCCESS"
            if failed == 0
            else "PARTIAL_SUCCESS"
        )

        print("\n==========================")
        print(" 테스트 적재 완료")
        print("==========================")
        print(f"조회     : {fetched}")
        print(f"신규     : {inserted}")
        print(f"수정     : {updated}")
        print(f"실패     : {failed}")

    except Exception as e:

        finished_at = datetime.now()

        print(
            f"\n[FATAL ERROR] {e}"
        )

        if conn:

            try:

                write_log(
                    conn,
                    started_at,
                    finished_at,
                    fetched,
                    inserted,
                    updated,
                    failed + 1,
                    "FAILED",
                    str(e)
                )

            except Exception:
                pass

    finally:

        if conn:
            conn.close()


if __name__ == "__main__":
    main()


[FATAL ERROR] (1045, "Access denied for user 'yoona'@'localhost' (using password: NO)")
